# When Can You Ship on Evals Alone? The Upworthy archive

This notebook reproduces the empirical illustration of the paper (Section 7.2 and
Appendix A.8) from the raw Upworthy Research Archive (exploratory packages file,
Matias et al. 2021) and a cache of LLM-judge ratings. It makes no API calls.

The archive records thousands of headline A/B tests run by Upworthy between 2013 and 2015.
Each test showed two or more headlines for the same article to randomized traffic and
recorded impressions and clicks. We treat each non-baseline headline as an intervention:
its online effect is the change in click-through rate, and its eval effect is the change
in an LLM judge's rating of whether the headline conveys the article's content without
withholding the point. The eval targets informativeness while the outcome is engagement,
which is what makes this a useful negative case.

| Notebook section | Where it appears in the paper |
|---|---|
| 1–4. Corpus construction | Section 7.2, first paragraph |
| 5. Judge scores | Section 7.2, second paragraph |
| 6. Reliability and surrogacy | Section 7.2, third paragraph |
| 7. The gate | Section 7.2, third paragraph; Appendix A.8 |

Everything is deterministic given the two seeds below. Results are written to
`results/upworthy_numbers.json`.

In [1]:
import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

DATA = Path("data")
OUT = Path("results"); OUT.mkdir(exist_ok=True)
SPLIT_SEED = 0         # binomial split of each arm's outcome data
BOOTSTRAP_SEED = 1     # bootstrap resampling of interventions
ALPHA = 0.10
NUMBERS = {}

## 1. Packages

A package is one headline arm of one test. Matias et al. (2021) document that
randomization failed for tests created between 25 June 2013 and 10 January 2014. The
exploratory file carries no flag for these tests, so we remove them by the creation date
of their earliest package.

In [2]:
packages = pd.read_csv(DATA / "upworthy-archive-exploratory-packages.csv", low_memory=False)
packages["created_at"] = pd.to_datetime(packages.created_at, format="mixed")
funnel = [("packages in the exploratory file", len(packages), packages.clickability_test_id.nunique())]

created = packages.groupby("clickability_test_id").created_at.min()
failed = created[(created >= "2013-06-25") & (created < "2014-01-11")].index
packages = packages[~packages.clickability_test_id.isin(failed)]
funnel.append(("after removing the randomization-failure window", len(packages),
               packages.clickability_test_id.nunique()))

packages = packages[(packages.impressions > 0) & packages.headline.notna()
                    & (packages.clicks <= packages.impressions)]
arms = packages.groupby("clickability_test_id").size()
packages = packages[packages.clickability_test_id.isin(arms[arms >= 2].index)].reset_index(drop=True)
funnel.append(("valid arms, at least two per test", len(packages), packages.clickability_test_id.nunique()))
for step, n_packages, n_tests in funnel:
    print(f"{step:50s} {n_packages:6d} packages {n_tests:5d} tests")
NUMBERS["tests_in_exploratory_file"] = funnel[0][2]
NUMBERS["tests_in_failure_window"] = len(failed)

packages in the exploratory file                    22666 packages  4873 tests
after removing the randomization-failure window     18388 packages  3823 tests
valid arms, at least two per test                   18388 packages  3823 tests


## 2. Interventions and the outcome split

The baseline of a test is its highest-impression arm, the most precisely measured one.
Every other arm is an intervention against it. Each arm's impressions are split
binomially in half and its clicks hypergeometrically in proportion, which gives two
independent outcome measurements per intervention, one used for fitting and one held out.
The held-out half would measure the gate's realized false-ship rate on independent noise.
In this illustration the gate ships nothing, so it is not used.

In [3]:
rng = np.random.default_rng(SPLIT_SEED)

def split_arm(impressions, clicks):
    n_fit = rng.binomial(int(impressions), 0.5)
    c_fit = rng.hypergeometric(int(clicks), int(impressions - clicks), n_fit)
    return n_fit, c_fit, int(impressions) - n_fit, int(clicks) - c_fit

rows = []
for test_id, arms_of_test in packages.groupby("clickability_test_id"):
    arms_of_test = arms_of_test.sort_values("impressions", ascending=False)
    base = arms_of_test.iloc[0]
    b_fit_n, b_fit_c, b_ev_n, b_ev_c = split_arm(base.impressions, base.clicks)
    for _, arm in arms_of_test.iloc[1:].iterrows():
        t_fit_n, t_fit_c, t_ev_n, t_ev_c = split_arm(arm.impressions, arm.clicks)
        rows.append(dict(test_id=test_id, headline_t=arm.headline, headline_b=base.headline,
                         t_fit_n=t_fit_n, t_fit_c=t_fit_c, t_ev_n=t_ev_n, t_ev_c=t_ev_c,
                         b_fit_n=b_fit_n, b_fit_c=b_fit_c, b_ev_n=b_ev_n, b_ev_c=b_ev_c))
interventions = pd.DataFrame(rows)

def difference_in_ctr_pp(c1, n1, c0, n0):
    p1, p0 = c1 / n1, c0 / n0
    return 100 * (p1 - p0), 100 * np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)

interventions["tauY"], interventions["seY"] = difference_in_ctr_pp(
    interventions.t_fit_c, interventions.t_fit_n, interventions.b_fit_c, interventions.b_fit_n)
interventions["tauY_holdout"], interventions["seY_holdout"] = difference_in_ctr_pp(
    interventions.t_ev_c, interventions.t_ev_n, interventions.b_ev_c, interventions.b_ev_n)
print(f"{len(interventions)} interventions (arm versus baseline)")

14565 interventions (arm versus baseline)


## 3. Intervention classes

A class names the kind of change, not the kind of headline. Each intervention is named
by the first of five text features that the treated headline adds or drops relative to its
baseline: a digit, a closing question mark, second-person address, an apostrophe or
quotation mark, and a length above 80 characters. The fourth feature mostly detects
apostrophes in contractions and possessives, since straight double quotes appear in only
about one headline in a hundred, and its class names say so. Classes with fewer than 60
interventions in the archive, and interventions that change none of the features, go to
`other`, which is not analyzed.

In [4]:
FEATURES = {
    "number":         lambda h: bool(re.search(r"\d", h)),
    "question":       lambda h: h.strip().endswith("?"),
    "second_person":  lambda h: bool(re.search(r"\b(you|your|yours)\b", h, re.I)),
    "apostrophe":     lambda h: ('"' in h) or ("'" in h),
    "long":           lambda h: len(h) > 80,
}
MIN_CLASS_SIZE = 60

def change_class(row):
    for name, has_feature in FEATURES.items():
        treated, baseline = has_feature(row.headline_t), has_feature(row.headline_b)
        if treated != baseline:
            return f"{name}_{'added' if treated else 'removed'}"
    return "other"

interventions["change"] = interventions.apply(change_class, axis=1)
sizes = interventions.change.value_counts()
interventions.loc[~interventions.change.isin(sizes[sizes >= MIN_CLASS_SIZE].index), "change"] = "other"
print(interventions.change.value_counts().to_string())

change
other                    9056
second_person_added       679
apostrophe_added          672
number_removed            662
second_person_removed     633
apostrophe_removed        618
number_added              607
question_removed          434
question_added            412
long_added                403
long_removed              389


## 4. One intervention per test, above an impression floor

Interventions that share a baseline arm share its sampling noise, which would violate the
independence across interventions in Assumption 2. We keep, for each test, the
intervention with the most impressions, and require at least 5,000 impressions on both
arms so that the outcome deltas are measured precisely.

In [5]:
MIN_IMPRESSIONS = 5000
total = interventions.t_fit_n + interventions.t_ev_n + interventions.b_fit_n + interventions.b_ev_n
one_per_test = interventions.loc[total.groupby(interventions.test_id).idxmax()]
precise = one_per_test[(one_per_test.t_fit_n + one_per_test.t_ev_n >= MIN_IMPRESSIONS)
                       & (one_per_test.b_fit_n + one_per_test.b_ev_n >= MIN_IMPRESSIONS)]
corpus = precise[precise.change != "other"].sort_values("test_id").reset_index(drop=True)
for step, frame in [("one intervention per test", one_per_test),
                    ("at least 5,000 impressions on both arms", precise),
                    ("in an analyzed class", corpus)]:
    print(f"{step:42s} {len(frame):5d}")
print()
print(corpus.change.value_counts().to_string())
NUMBERS.update(n_interventions=len(corpus), n_classes=corpus.change.nunique(),
               class_sizes=corpus.change.value_counts().to_dict())

one intervention per test                   3823
at least 5,000 impressions on both arms      489
in an analyzed class                         183

change
number_added             27
second_person_removed    25
second_person_added      21
number_removed           20
question_added           18
apostrophe_added         18
question_removed         16
apostrophe_removed       16
long_removed             13
long_added                9


## 5. Eval effects from the LLM judge

The judge (GPT-4o-mini, temperature 1) rated each headline on a 0–100 scale against one
assertion, five times. A headline's eval score is the mean of its ratings rescaled to
[0, 1], with standard error $s/\sqrt K$, and an intervention's eval effect is the difference
between the treated and baseline scores in percentage points.

The cache maps `sha1("openai|gpt-4o-mini|5|<assertion>|<headline>")` to the list of five
ratings. The next cell checks that every headline in the corpus is in the cache before
scoring anything.

In [6]:
ASSERTION = ("The headline accurately and clearly conveys what the reader will "
             "get from the article, without withholding the point to force a click.")
JUDGE_MODEL, K_RATINGS = "gpt-4o-mini", 5
cache = json.loads((DATA / "judge_cache.json").read_text())

def cache_key(headline):
    return hashlib.sha1(f"openai|{JUDGE_MODEL}|{K_RATINGS}|{ASSERTION}|{headline}".encode()).hexdigest()

headlines = pd.unique(pd.concat([corpus.headline_t, corpus.headline_b]))
missing = [h for h in headlines if cache_key(h) not in cache]
if missing:
    raise RuntimeError(f"{len(missing)} of {len(headlines)} corpus headlines have no cached judge ratings, "
                       f"for example {missing[:3]}. Rerun the judge for these headlines and add their "
                       "ratings to data/judge_cache.json under the key described above.")

def judge_score(headline):
    ratings = np.asarray(cache[cache_key(headline)], float) / 100.0
    return ratings.mean(), ratings.std(ddof=1) / np.sqrt(len(ratings))

scores = {h: judge_score(h) for h in headlines}
q_t, s_t = np.array([scores[h] for h in corpus.headline_t]).T
q_b, s_b = np.array([scores[h] for h in corpus.headline_b]).T
corpus["tauS"], corpus["seS"] = 100 * (q_t - q_b), 100 * np.sqrt(s_t**2 + s_b**2)
print(f"{len(headlines)} unique headlines, all scored from the cache")

366 unique headlines, all scored from the cache


## 6. Reliability and the moment-corrected estimate

The reliability ratio of each side is its between-intervention variance divided by its
total observed variance, with the between-intervention variance estimated by subtracting
the mean squared standard error. The corrected covariance is
$\hat\Omega=\Pi_{\succeq0}(W_n-\bar V_n)$ (Theorem 3). Everything below is computed for a
whole batch of corpora at once, so the bootstrap is a single vectorized call.

In [7]:
def moment_estimates(x, y, var_s, var_y):
    # x, y, var_s, var_y have shape (batch, n); returns the corrected estimates per corpus
    n = x.shape[-1]
    xc, yc = x - x.mean(-1, keepdims=True), y - y.mean(-1, keepdims=True)
    w = np.stack([np.stack([(xc * xc).sum(-1), (xc * yc).sum(-1)], -1),
                  np.stack([(xc * yc).sum(-1), (yc * yc).sum(-1)], -1)], -2) / (n - 1)
    v_bar = np.stack([var_s.mean(-1), var_y.mean(-1)], -1)
    raw = w - v_bar[..., :, None] * np.eye(2)
    eigval, eigvec = np.linalg.eigh(raw)
    om = (eigvec * np.maximum(eigval, 0)[..., None, :]) @ np.swapaxes(eigvec, -1, -2)
    den = om[..., 0, 0] * om[..., 1, 1]
    return dict(W=w, Om=om, v_bar=v_bar, projected=(eigval < 0).any(-1),
                naive_R2=w[..., 0, 1] ** 2 / (w[..., 0, 0] * w[..., 1, 1]),
                R2=np.where(den > 0, om[..., 0, 1] ** 2 / np.where(den > 0, den, 1), np.nan),
                mu=np.stack([x.mean(-1), y.mean(-1)], -1))

def columns(df, idx=None):
    cols = [df.tauS.to_numpy(), df.tauY.to_numpy(), df.seS.to_numpy() ** 2, df.seY.to_numpy() ** 2]
    return [c[None, :] if idx is None else c[idx] for c in cols]

pooled = {k: v[0] for k, v in moment_estimates(*columns(corpus)).items()}
kappa_s = 1 - pooled["v_bar"][0] / pooled["W"][0, 0]
kappa_y = 1 - pooled["v_bar"][1] / pooled["W"][1, 1]
a = pooled["Om"][0, 0] + pooled["v_bar"][0]
residual_variance = pooled["Om"][1, 1] - pooled["Om"][0, 1] ** 2 / a
print(f"reliability: eval {kappa_s:.3f}, outcome {kappa_y:.3f}, product {kappa_s * kappa_y:.3f}")
print(f"naive R2 {pooled['naive_R2']:.4f}, corrected R2 {pooled['R2']:.4f}, "
      f"corrected Omega_SY {pooled['Om'][0, 1]:+.4f}, projection needed: {pooled['projected']}")
print(f"outcome noise / residual variance: {pooled['v_bar'][1] / residual_variance:.3f}")

reliability: eval 0.931, outcome 0.735, product 0.685
naive R2 0.0161, corrected R2 0.0234, corrected Omega_SY +0.9500, projection needed: False
outcome noise / residual variance: 0.368


### Bootstrap over interventions

Percentile intervals for the corrected $R^2_{\mathcal C}$ and for $\Omega_{SY}$, from 2,000
resamples of interventions. $R^2_{\mathcal C}=0$ exactly when $\Omega_{SY}=0$, and unlike
$\hat R^2_{\mathcal C}$ the covariance has a regular normal limit at zero (Section 4 of the
paper), so significance is judged from the interval for $\Omega_{SY}$. The share of resamples
in which the noise subtraction had to be projected is a diagnostic for the correction.

In [8]:
RESAMPLES = 2000

def bootstrap(df, seed):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(df), (RESAMPLES, len(df)))
    e = moment_estimates(*columns(df, idx))
    return dict(R2_ci=np.nanpercentile(e["R2"], [2.5, 97.5]),
                osy_ci=np.percentile(e["Om"][:, 0, 1], [2.5, 97.5]),
                projection_rate=e["projected"].mean())

boot = bootstrap(corpus, BOOTSTRAP_SEED)
print(f"corrected R2 {pooled['R2']:.3f}, 95% interval [{boot['R2_ci'][0]:.3f}, {boot['R2_ci'][1]:.3f}]")
print(f"Omega_SY 95% interval [{boot['osy_ci'][0]:+.3f}, {boot['osy_ci'][1]:+.3f}]")
print(f"resamples needing projection: {100 * boot['projection_rate']:.1f}%")

corrected R2 0.023, 95% interval [0.000, 0.126]
Omega_SY 95% interval [-0.516, +2.535]
resamples needing projection: 0.0%


### Per class

Class samples are small, so these estimates are exploratory.

In [9]:
per_class = []
for i, (change, rows_of_class) in enumerate(corpus.groupby("change")):
    e = {k: v[0] for k, v in moment_estimates(*columns(rows_of_class)).items()}
    b = bootstrap(rows_of_class, BOOTSTRAP_SEED + 1 + i)
    per_class.append(dict(change=change, n=len(rows_of_class), naive_R2=e["naive_R2"], R2=e["R2"],
                          osy=e["Om"][0, 1], osy_low=b["osy_ci"][0], osy_high=b["osy_ci"][1],
                          interval_excludes_zero=not (b["osy_ci"][0] <= 0 <= b["osy_ci"][1]),
                          projected=bool(e["projected"])))
per_class = pd.DataFrame(per_class)
print(per_class.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

               change  n  naive_R2     R2     osy  osy_low  osy_high  interval_excludes_zero  projected
     apostrophe_added 18    0.0282 0.0404  1.3883  -1.9575    5.5404                   False      False
   apostrophe_removed 16    0.0998 0.1315  2.7694  -2.3973    9.6017                   False      False
           long_added  9    0.0220 0.1524  0.8225  -2.2428    3.8559                   False      False
         long_removed 13    0.0022 0.0074  0.1869  -2.3254    2.8635                   False      False
         number_added 27    0.0408 0.1042 -0.7527  -1.9963    0.3691                   False      False
       number_removed 20    0.0004 0.0009 -0.1150  -2.0937    1.9951                   False      False
       question_added 18    0.0430 0.0553  1.8921  -2.1860    6.2563                   False      False
     question_removed 16    0.0153 0.0189  2.0287  -7.3962   12.4677                   False      False
  second_person_added 21    0.0009 0.0014 -0.2020  -2.4609    2.

## 7. The gate

The pooled plug-in gate at $\alpha=0.10$ needs a positive fitted covariance and a positive
residual variance. Otherwise no positive-slope gate exists, the threshold is infinite, and
every intervention is routed to an A/B test.

In [10]:
if pooled["Om"][0, 1] > 0 and residual_variance > 0:
    s_star = pooled["mu"][0] + (stats.norm.ppf(1 - ALPHA) * np.sqrt(residual_variance)
                                - pooled["mu"][1]) * a / pooled["Om"][0, 1]
    gate_status = f"open, s* = {s_star:.2f} pp, ships {(corpus.tauS >= s_star).sum()} interventions"
else:
    gate_status = "closed: no positive-slope gate exists, every intervention goes to an A/B test"
print(gate_status)

open, s* = 133.83 pp, ships 0 interventions


In [11]:
def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

NUMBERS.update(
    kappa_S=kappa_s, kappa_Y=kappa_y, kappa_product=kappa_s * kappa_y,
    naive_R2=pooled["naive_R2"], corrected_R2=pooled["R2"], corrected_R2_interval=boot["R2_ci"],
    corrected_osy=pooled["Om"][0, 1], osy_interval=boot["osy_ci"],
    projection_needed=pooled["projected"], bootstrap_projection_rate=boot["projection_rate"],
    noise_to_residual=pooled["v_bar"][1] / residual_variance, gate=gate_status,
    per_class=per_class.to_dict(orient="records"))
(OUT / "upworthy_numbers.json").write_text(json.dumps(to_jsonable(NUMBERS), indent=2))
for key, val in NUMBERS.items():
    if key != "per_class":
        print(f"{key:28s} {to_jsonable(val)}")

tests_in_exploratory_file    4873
tests_in_failure_window      1050
n_interventions              183
n_classes                    10
class_sizes                  {'number_added': 27, 'second_person_removed': 25, 'second_person_added': 21, 'number_removed': 20, 'question_added': 18, 'apostrophe_added': 18, 'question_removed': 16, 'apostrophe_removed': 16, 'long_removed': 13, 'long_added': 9}
kappa_S                      0.9313361934364929
kappa_Y                      0.7352106668052172
kappa_product                0.6847283037962767
naive_R2                     0.016056311927476634
corrected_R2                 0.023449172231463318
corrected_R2_interval        [8.762050367150407e-05, 0.12567443293521002]
corrected_osy                0.9500488598069706
osy_interval                 [-0.5155666620573949, 2.5345141166417733]
projection_needed            False
bootstrap_projection_rate    0.0
noise_to_residual            0.3681954109000676
gate                         open, s* = 133.83 pp, sh